# A Sentinel-2 to AquaCrop-OSPy Pipeline for Table Grapes in the Hex River Valley

**Completed Honours research | Dagbreek / Modderdrift, Western Cape | Seven growing seasons**

This project uses satellite and weather data to estimate seasonal grapevine growth and compare two crop-model configurations.

**Follow the numbered sections from 1 to 13.** Sections 1–12 contain the full implementation. Section 13 displays saved results for the video demo, without rerunning AquaCrop.

**Important:** The satellite greenness curve is not measured vine canopy cover. Error improvements show *in-sample* model–satellite agreement, not field validation or proven irrigation savings.

**To run everything:** Keep the complete repository folder, install `requirements.txt`, then use **Run All**. The default satellite source uses archived data. If you only want to show completed results, jump to **Section 13** and run its cells.

## 1. Setup

Find the repository, load the helper functions and choose the data source.

### 1.1 Upload and extract the project

In Colab, upload **`Sentinel2_AquaCropOSPy_TableGrapes_HexRiver_Project.zip`** using the Files panel. This cell finds the complete project and makes its data and Python helpers available. Run it once per new Colab session.

In [ ]:
# 1.1 — Find or extract the research project
import os
import sys
import zipfile
from pathlib import Path

PROJECT_FOLDER = "sentinel2-aquacrop-hex-river-final"
WORK_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()

# Reuse an existing extracted project (works in Colab or locally).
locations = [WORK_DIR / PROJECT_FOLDER, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (folder.resolve() for folder in locations
     if (folder / "src" / "pipeline_utils.py").is_file()),
    None,
)

if PROJECT_ROOT is None:
    # Look for an uploaded ZIP containing the complete project structure.
    project_zips = sorted(WORK_DIR.glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    required_member = f"{PROJECT_FOLDER}/src/pipeline_utils.py"
    for zip_path in project_zips:
        with zipfile.ZipFile(zip_path) as archive:
            if required_member not in archive.namelist():
                continue  # This may be the outer download ZIP, not the project ZIP.
            archive.extractall(WORK_DIR)
        PROJECT_ROOT = (WORK_DIR / PROJECT_FOLDER).resolve()
        print("Extracted project ZIP:", zip_path.name)
        break

if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "pipeline_utils.py").is_file():
    raise FileNotFoundError(
        "Complete project not found. Upload the *Project.zip file* from your "
        "download package into Colab's /content folder, then rerun this cell."
    )

# Make the research data and helper module discoverable by the next cells.
os.environ["PIPELINE_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project ready:", PROJECT_ROOT)
print("Helper module:", PROJECT_ROOT / "src" / "pipeline_utils.py")

### 1.2 Install required libraries

Install dependencies from the project's `requirements.txt` before importing AquaCrop and RefET. In a new Colab session, run this cell once.

In [ ]:
# 1.2 — Install the project libraries
requirements_file = str(PROJECT_ROOT / "requirements.txt")
%pip install -q -r {requirements_file}

In [ ]:
# 1.3 — Load libraries, files and settings
import copy
import hashlib
import json
import os
import platform
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set PIPELINE_ROOT if the notebook is opened from outside the repository.
candidates = ([Path(os.environ["PIPELINE_ROOT"])] if os.environ.get("PIPELINE_ROOT") else []) + [
    Path.cwd(), Path.cwd().parent, Path("/content/sentinel2-aquacrop-hex-river-final")
]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src" / "pipeline_utils.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Repository not found. Clone the complete folder, then set PIPELINE_ROOT to its path.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.pipeline_utils import (
    SEASONS, FEATURES, check_seasons, clean_satellite_features,
    canopy_proxy_and_smoothing, phenology_from_curves, event_gdd_table,
    daily_gdd, aquacrop_max_canopy_gdd, derive_cgc,
    metrics, normalized_metrics, safe_pct_change,
)
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
STUDY_AREA = PROJECT_ROOT / "data" / "study_area"
TABLES = PROJECT_ROOT / "outputs" / "tables"
FIGS = {kind: PROJECT_ROOT / "outputs" / "figures" / kind for kind in ("canopy", "phenology", "aquacrop", "water")}
for directory in (PROCESSED, TABLES, *FIGS.values()):
    directory.mkdir(parents=True, exist_ok=True)
WEATHER_FILE = RAW / "weatherDatasetNASAPower.csv"
SATELLITE_ARCHIVE = RAW / "sentinel2_dagbreek_features_archived_219.csv"
SATELLITE_SOURCE = os.environ.get("SATELLITE_SOURCE", "cache").lower()
GEE_PROJECT = os.environ.get("GEE_PROJECT")
SCENE_CLOUD_THRESHOLD = 30
EE_START = "2017-01-01"
EE_END_EXCLUSIVE = "2024-05-01"  # Include 30 April 2024.
T_BASE, T_UPPER = 10.0, 35.0
MIN_EMERGENCE_GDD = 10.0  # Numerical safeguard, NOT grapevine biology.

print("Repository:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("Satellite source:", SATELLITE_SOURCE)
assert WEATHER_FILE.exists(), f"Missing NASA POWER source: {WEATHER_FILE}"
assert SATELLITE_SOURCE in ("cache", "earth_engine")

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


## 2. Weather data

Check NASA POWER weather records and prepare the daily AquaCrop input file.

### 2.1 Read and check daily weather

Reject missing fields and invalid or incomplete dates rather than guessing values.

In [ ]:
# 2.1 — Read and check daily weather
import refet
import aquacrop
print("AquaCrop version:", getattr(aquacrop, "__version__", "not exposed"))
print("RefET version:", getattr(refet, "__version__", "not exposed"))

raw_weather = pd.read_csv(WEATHER_FILE, skiprows=16, na_values=[-999, -999.0])
required_power = ["YEAR", "DOY", "T2M_MIN", "T2M_MAX", "PRECTOTCORR", "ALLSKY_SFC_SW_DWN", "WS2M", "T2MDEW"]
missing_columns = set(required_power) - set(raw_weather.columns)
if missing_columns:
    raise ValueError(f"NASA POWER columns missing: {sorted(missing_columns)}")
if raw_weather[required_power].isna().any().any():
    raise ValueError("Missing/-999 values in required NASA POWER variables. Do not silently interpolate.")
weather = raw_weather.rename(columns={
    "ALLSKY_SFC_SW_DWN": "SolarRadiation", "T2M_MAX": "MaxTemp", "T2M_MIN": "MinTemp",
    "T2MDEW": "DewPoint", "PRECTOTCORR": "Precipitation", "RH2M": "RelativeHumidity",
    "WS2M": "WindSpeed", "GWETROOT": "RootMoisture", "DOY": "DayOfYear",
}).copy()
weather["Date"] = pd.to_datetime(weather["YEAR"].astype(int).astype(str) + "-01-01") + pd.to_timedelta(weather["DayOfYear"] - 1, unit="D")
weather = weather.sort_values("Date").reset_index(drop=True)
if not weather["Date"].is_unique or not weather["Date"].diff().dropna().eq(pd.Timedelta(days=1)).all():
    raise ValueError("NASA POWER weather dates are duplicated or not consecutive daily dates")
if (weather["MaxTemp"] < weather["MinTemp"]).any() or (weather["Precipitation"] < 0).any():
    raise ValueError("Physically implausible weather input")


### 2.2 Calculate reference evapotranspiration

Reference evapotranspiration (ET₀) describes the atmosphere’s demand for water.

In [ ]:
# 2.2 — Calculate reference evapotranspiration
# Calculate reference evapotranspiration for all weather days
# using the ASCE standardized short-reference method.

eto_model = refet.Daily(
    tmin=weather["MinTemp"].to_numpy(dtype=float),
    tmax=weather["MaxTemp"].to_numpy(dtype=float),
    rs=weather["SolarRadiation"].to_numpy(dtype=float),
    uz=weather["WindSpeed"].to_numpy(dtype=float),
    zw=2,
    elev=722.15,
    lat=-33.6423,
    doy=weather["DayOfYear"].to_numpy(dtype=int),
    tdew=weather["DewPoint"].to_numpy(dtype=float),
    method="asce"
)

eto_values = np.asarray(eto_model.eto(), dtype=float)

# Confirm one ET value was calculated for every weather day.
if eto_values.shape != (len(weather),):
    raise ValueError(
        f"Unexpected ReferenceET shape: {eto_values.shape}"
    )

weather["ReferenceET"] = eto_values

print("ReferenceET calculated successfully!")
print(weather[["Date", "ReferenceET"]].head())
if not np.isfinite(weather["ReferenceET"]).all() or (weather["ReferenceET"] < 0).any():
    raise ValueError("ReferenceET contains invalid values")
weather.to_csv(PROCESSED / "nasa_power_weather_processed.csv", index=False)
weather_aqua = pd.DataFrame({
    "Day": weather.Date.dt.day, "Month": weather.Date.dt.month, "Year": weather.Date.dt.year,
    "MinTemp": weather.MinTemp, "MaxTemp": weather.MaxTemp,
    "Precipitation": weather.Precipitation, "ReferenceET": weather.ReferenceET,
})
weather_aqua.to_csv(PROCESSED / "aquacrop_weather.txt", sep=" ", index=False)
print(f"Weather: {len(weather)} days, {weather.Date.min().date()}–{weather.Date.max().date()}")
print(weather[["ReferenceET", "MinTemp", "MaxTemp", "Precipitation"]].describe().round(2))


## 3. Sentinel-2 observations

Read vineyard satellite data and check that the seven seasons are present.

### 3.1 Load the vineyard and satellite observations

Use the saved Sentinel-2 extraction or explicitly choose Earth Engine.

In [ ]:
# 3.1 — Load the vineyard and satellite observations
import geopandas as gpd
import json
from shapely.geometry import shape
geojson = json.loads((STUDY_AREA / "dagbreek_vineyard_block.geojson").read_text())
geom = shape(geojson["features"][0]["geometry"])
if not geom.is_valid:
    raise ValueError("Dagbreek polygon is not valid")
poly_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326")
print("Dagbreek polygon area (ha):", round(float(poly_gdf.to_crs("EPSG:32734").area.iloc[0] / 10000), 3))

if SATELLITE_SOURCE == "cache":
    assert SATELLITE_ARCHIVE.exists(), f"Missing satellite cache: {SATELLITE_ARCHIVE}"
    incoming_satellite = pd.read_csv(SATELLITE_ARCHIVE)
    selected_satellite_path = SATELLITE_ARCHIVE
    print("Using archived research extraction:", selected_satellite_path.name)
else:
    if not GEE_PROJECT:
        raise ValueError("Set a valid GEE_PROJECT environment variable before Earth Engine re-extraction")
    import ee
    import geemap
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
    polygon_ee = ee.Geometry.Polygon([list(map(list, geom.exterior.coords))])

    def mask_clouds(image):
        scl = image.select("SCL")
        clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                 .And(scl.neq(10)).And(scl.neq(11)))
        return image.updateMask(clear).copyProperties(image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])

    def add_indices(image):
        scaled = image.select(["B4", "B5", "B8", "B11"]).multiply(.0001)
        image = image.addBands(scaled, overwrite=True)
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
        evi2 = image.expression("2.5*(NIR-RED)/(NIR+2.4*RED+1)",
                                {"NIR":image.select("B8"), "RED":image.select("B4")}).rename("EVI2")
        ndmi = image.normalizedDifference(["B8", "B11"]).rename("NDMI")
        ndre = image.normalizedDifference(["B8", "B5"]).rename("NDRE")
        return image.addBands([ndvi,evi2,ndmi,ndre])

    def extract_block_means(image):
        vals = image.select(list(FEATURES)).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=polygon_ee, scale=20, maxPixels=1e9)
        vals = vals.set("Date", ee.Date(image.get("system:time_start")).format("YYYY-MM-dd"))
        return ee.Feature(None, vals)

    full_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                       .filterBounds(polygon_ee).filterDate(EE_START, EE_END_EXCLUSIVE))
    cloud_filtered = full_collection.filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", SCENE_CLOUD_THRESHOLD))
    print("Before cloud filter:", full_collection.size().getInfo())
    print("After cloud filter:", cloud_filtered.size().getInfo())
    features_ee = ee.FeatureCollection(cloud_filtered.map(mask_clouds).map(add_indices).map(extract_block_means))
    incoming_satellite = geemap.ee_to_df(features_ee)
    # Save raw export under new name. Do not erase original 219-date archive.
    selected_satellite_path = RAW / "sentinel2_dagbreek_features_reextracted.csv"
    incoming_satellite.to_csv(selected_satellite_path, index=False)
    print("GEE export saved separately:", selected_satellite_path.name)


### 3.2 Clean and count the satellite dates

Confirm the records cover all seven September–April growing seasons.

In [ ]:
# 3.2 — Clean and count the satellite dates
satellite = clean_satellite_features(incoming_satellite)
check_seasons(satellite)
satellite.to_csv(PROCESSED / "sentinel2_features_clean.csv", index=False)
print("Selected observations:", len(satellite))
print(satellite.groupby("Season").size().to_string())
print("Input SHA256:", file_sha256(selected_satellite_path))


## 4. Vegetation greenness

Use NDVI to describe the seasonal vegetation pattern. This is a greenness proxy, not measured vine canopy cover.

### 4.1 Build the greenness proxy

Scale NDVI using pooled 5th and 95th percentiles; smooth the series every five days.

In [ ]:
# 4.1 — Build the greenness proxy
satellite_cc, canopy_smoothed, observation_gaps, canopy_settings = canopy_proxy_and_smoothing(
    satellite, interval_days=5, window=7, polyorder=2)
satellite_cc.to_csv(PROCESSED / "sentinel2_features_with_cc_proxy.csv", index=False)
canopy_smoothed.to_csv(PROCESSED / "sentinel2_canopy_smoothed.csv", index=False)
observation_gaps.to_csv(TABLES / "observation_gap_qa.csv", index=False)
pd.DataFrame([canopy_settings]).to_csv(TABLES / "canopy_proxy_parameters.csv", index=False)
print("NDVI low/high:", canopy_settings["NDVI_low_endmember"], canopy_settings["NDVI_high_endmember"])
print(observation_gaps[["Season", "Number_of_Raw_Observations", "Median_Gap_Days", "Maximum_Gap_Days", "Gaps_Above_15_Days"]].to_string(index=False))


### 4.2 Plot the seasonal greenness curves

Save a raw-versus-smoothed graph for each growing season.

In [ ]:
# 4.2 — Plot the seasonal greenness curves
for season in SEASONS:
    raw = satellite_cc[satellite_cc.Season == season]
    curve = canopy_smoothed[canopy_smoothed.Season == season]
    fig, ax = plt.subplots(figsize=(10,4.6))
    ax.scatter(raw.Date, raw.Sentinel_CC_Proxy, s=26, label="Direct satellite observations")
    ax.plot(curve.Date, curve.Sentinel_CC_Smoothed, label="5-day reconstructed/smoothed proxy")
    ax.set(xlabel="Date", ylabel="NDVI-derived canopy proxy", title=f"Dagbreek canopy proxy: {season}", ylim=(0,1.05))
    ax.legend(); ax.grid(alpha=.2); fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(FIGS["canopy"] / f"raw_vs_smoothed_{season.replace('/','-')}.png", dpi=180)
    if season == "2023/2024": plt.show()
    else: plt.close(fig)


## 5. Growth-stage dates

Find the start, peak and decline of seasonal vegetation growth.

### 5.1 Detect growth stages

Extract seasonal green-up, maximum greenness and decline dates.

In [ ]:
# 5.1 — Detect growth stages
phenology = phenology_from_curves(canopy_smoothed, satellite_cc)
phenology.to_csv(PROCESSED / "phenology.csv", index=False)
required_events = ["Budbreak_Date", "Full_Budbreak_Date", "Maximum_CC_Date", "Senescence_Onset_Date"]
missing_essential = phenology[phenology[required_events].isna().any(axis=1)]
if len(missing_essential):
    raise ValueError("Essential phenological proxies missing: " + ", ".join(missing_essential.Season))
print(phenology[["Season", *required_events, "End_of_Season_Date", "Budbreak_To_FullBudbreak_Days"]].to_string(index=False))


### 5.2 Plot the growth-stage markers

Show where the estimated event dates fall on the greenness curves.

In [ ]:
# 5.2 — Plot the growth-stage markers
colors = {"Budbreak_Date":"green", "Full_Budbreak_Date":"blue", "Maximum_CC_Date":"darkorange",
          "Senescence_Onset_Date":"red", "End_of_Season_Date":"purple"}
for season in SEASONS:
    curve = canopy_smoothed[canopy_smoothed.Season == season]
    raw = satellite_cc[satellite_cc.Season == season]
    row = phenology[phenology.Season == season].iloc[0]
    fig, ax = plt.subplots(figsize=(11,4.8))
    ax.plot(curve.Date, curve.Sentinel_CC_Smoothed, lw=2, label="Smoothed satellite canopy proxy")
    ax.scatter(raw.Date, raw.Sentinel_CC_Proxy, s=24, alpha=.6, label="Original observations")
    for event, color in colors.items():
        if pd.notna(row[event]): ax.axvline(row[event], color=color, linestyle="--", label=event.replace("_Date", "").replace("_", " "))
    ax.set(xlabel="Date", ylabel="Canopy proxy (not physical vine CC)", title=f"Sentinel phenology proxies: {season}", ylim=(0,1.05))
    ax.legend(fontsize=8); ax.grid(alpha=.2); fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(FIGS["phenology"] / f"phenology_{season.replace('/','-')}.png", dpi=180)
    if season == "2023/2024": plt.show()
    else: plt.close(fig)


## 6. Growing degree days

Turn growth-stage dates into accumulated temperature units (GDD).

### 6.1 Calculate accumulated temperature

Use a 10°C base and a 35°C cap, matching the completed experiment.

In [ ]:
# 6.1 — Calculate accumulated temperature
phenology_gdd, daily_season_gdd = event_gdd_table(phenology, weather)
phenology_gdd.to_csv(PROCESSED / "phenology_gdd.csv", index=False)
daily_season_gdd.to_csv(PROCESSED / "daily_gdd.csv", index=False)
print(phenology_gdd[["Season", "Full_Budbreak_GDD", "Maximum_CC_GDD", "Senescence_Onset_GDD"]].round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(10.5,5))
ax.plot(phenology_gdd.Season, phenology_gdd.Maximum_CC_GDD, marker="o", label="Satellite maximum-canopy proxy")
ax.plot(phenology_gdd.Season, phenology_gdd.Senescence_Onset_GDD, marker="o", label="Satellite senescence proxy")
# The baseline reference lines are added once the AquaCrop crop object is constructed.
ax.set(xlabel="Season", ylabel="GDD from satellite-derived budbreak", title="Sentinel-derived thermal phenology")
ax.tick_params(axis="x", rotation=30); ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(FIGS["phenology"] / "phenology_gdd_by_season.png", dpi=180)
plt.show()


## 7. Crop model settings

Create the literature baseline and seven Sentinel-informed grapevine configurations.

### 7.1 Set up the literature baseline

Use the published grapevine approach. The Kcb=1.10 setting still needs primary-source verification.

In [ ]:
# 7.1 — Set up the literature baseline
from aquacrop import AquaCropModel, Crop, Soil, InitialWaterContent, IrrigationManagement
from aquacrop.utils import prepare_weather

grape_parameters = {
    "CropType": 3, "PlantMethod": 1, "CalendarType": 2, "GDDmethod": 1,
    "Tbase": T_BASE, "Tupp": T_UPPER,
    "PlantPop": 10000, "SeedSize": 640,  # numerical CC0 = .064
    "CCx": .65, "CGC": .0054, "CDC": .0014,
    "Emergence": 130., "Senescence": 1941., "Maturity": 3681.,
    "Zmax": 1.2, "WP": 35., "HI0": .55,
    "Kcb": 1.10,  # Explicit project-intended setting; check primary paper provenance.
    "p_up1": .30, "p_lo1": .80, "fshape_w1": 5.5,
    "p_up2": .50, "p_lo2": 1.00, "fshape_w2": 3.0,
    "p_up3": .85, "p_lo3": 1.00, "fshape_w3": 3.0,
    "p_up4": .90, "p_lo4": 1.00, "fshape_w4": 1.0,
}
# Set actual planting date separately for each season; this object verifies fixed parameters only.
literature_crop_check = Crop("custom", planting_date="10/07", **grape_parameters)
assert abs(literature_crop_check.CC0 - .064) < 1e-8
assert abs(literature_crop_check.Kcb - 1.10) < 1e-8
baseline_max_gdd = aquacrop_max_canopy_gdd(literature_crop_check)
print("Model-derived literature baseline maximum canopy:", round(baseline_max_gdd,2), "GDD")
print("Explicit baseline Kcb:", literature_crop_check.Kcb)


### 7.2 Create Sentinel-informed crops

Use season-specific timing. Aligning peak timing is part of the configuration, not independent validation.

In [ ]:
# 7.2 — Create Sentinel-informed crops
sentinel_crops = {}
parameter_rows = []
for row in phenology_gdd.itertuples(index=False):
    if not np.isfinite([row.Full_Budbreak_GDD, row.Maximum_CC_GDD, row.Senescence_Onset_GDD]).all():
        raise ValueError(f"{row.Season}: a required satellite timing value is missing")
    # A 10-GDD floor prevents a numerical failure; it is not a biological threshold.
    emergence_sim = max(float(row.Full_Budbreak_GDD), MIN_EMERGENCE_GDD)
    cgcs = derive_cgc(literature_crop_check.CC0, literature_crop_check.CCx, emergence_sim, float(row.Maximum_CC_GDD))
    params = grape_parameters.copy()
    params.update(Emergence=emergence_sim, CGC=cgcs, Senescence=float(row.Senescence_Onset_GDD))
    bud = pd.Timestamp(row.Budbreak_Date)
    informed = Crop("custom", planting_date=bud.strftime("%m/%d"), **params)
    calc_peak = aquacrop_max_canopy_gdd(informed)
    assert abs(calc_peak - float(row.Maximum_CC_GDD)) < .0001
    sentinel_crops[row.Season] = informed
    parameter_rows.append({
        "Season":row.Season, "Raw_FullBudbreak_GDD":row.Full_Budbreak_GDD,
        "Simulation_Emergence_GDD":emergence_sim, "Sentinel_Peak_GDD":row.Maximum_CC_GDD,
        "Derived_CGC":cgcs, "Calculated_Model_Peak_GDD":calc_peak,
        "Sentinel_Senescence_GDD":row.Senescence_Onset_GDD,
        "Numerical_Emergence_Floor_Used":bool(row.Full_Budbreak_GDD < MIN_EMERGENCE_GDD),
    })
parameter_table = pd.DataFrame(parameter_rows)
parameter_table.to_csv(TABLES / "sentinel_informed_crop_parameters.csv", index=False)
print(parameter_table.round(3).to_string(index=False))


### 7.3 Compare crop-development dates

Record differences between the baseline timing and satellite-derived timing.

In [ ]:
# 7.3 — Compare crop-development dates
# Add internally computed baseline reference GDD lines for transparent comparison.
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(phenology_gdd.Season, phenology_gdd.Maximum_CC_GDD, marker="o", label="Satellite peak proxy")
ax.plot(phenology_gdd.Season, phenology_gdd.Senescence_Onset_GDD, marker="o", label="Satellite senescence proxy")
ax.axhline(baseline_max_gdd, color="gray", ls="--", label="Baseline internally derived peak")
ax.axhline(grape_parameters["Senescence"], color="black", ls=":", label="Baseline senescence")
ax.set(xlabel="Growing season", ylabel="GDD from satellite budbreak", title="Literature-baseline vs satellite-derived thermal timing")
ax.tick_params(axis="x", rotation=30); ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(FIGS["phenology"] / "phenology_gdd_vs_literature_baseline.png", dpi=180)
plt.show()


## 8. Soil and irrigation scenario

Use the same soil, initial water and irrigation assumptions for each model pair.

### 8.1 Set the shared modelling scenario

The soil and irrigation inputs are assumed scenarios, not field measurements.

In [ ]:
# 8.1 — Set the shared modelling scenario
soil_baseline = Soil(soil_type="SandyLoam")
initial_water = InitialWaterContent(value=["FC"])
well_watered = IrrigationManagement(irrigation_method=1, SMT=[100, 100, 100, 100])
print("Soil:", soil_baseline.Name, "Depth (m):", soil_baseline.zSoil)
print("Initial soil-water condition: FC at simulation start")
print("Irrigation SMT:", well_watered.SMT)


## 9. Run the simulations

Run both configurations for each of the seven seasons and save the daily outputs.

### 9.1 Prepare model inputs and safety checks

The helper functions enforce complete and correctly ordered daily outputs.

In [ ]:
# 9.1 — Prepare model inputs and safety checks
weather_df = prepare_weather(str(PROCESSED / "aquacrop_weather.txt"))
weather_df["Date"] = pd.to_datetime(weather_df.Date)
weather_df = weather_df.sort_values("Date").reset_index(drop=True)
weather_df["Daily_GDD"] = daily_gdd(weather_df.MinTemp, weather_df.MaxTemp, T_BASE, T_UPPER)

def first_date_after_target_gdd(budbreak, threshold):
    future = weather_df[(weather_df.Date >= budbreak) & (weather_df.Date <= budbreak + pd.Timedelta(days=350))].copy()
    future["CumGDD"] = future.Daily_GDD.cumsum()
    matches = future[future.CumGDD > threshold]
    if matches.empty:
        raise ValueError(f"Unable to find a terminal date beyond {threshold} GDD for {budbreak}")
    return pd.Timestamp(matches.iloc[0].Date)

def run_model_pair(sim_start, sim_end, crop):
    model = AquaCropModel(
        sim_start_time=sim_start.strftime("%Y/%m/%d"),
        sim_end_time=sim_end.strftime("%Y/%m/%d"),
        weather_df=weather_df.drop(columns=["Daily_GDD"]),
        soil=copy.deepcopy(soil_baseline),
        crop=copy.deepcopy(crop),
        initial_water_content=copy.deepcopy(initial_water),
        irrigation_management=copy.deepcopy(well_watered),
        off_season=True,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*FreshYield.*")
        model.run_model(till_termination=True)
    return model

def dated_output(model, getter, sim_start, required_end):
    table = getattr(model, getter)().copy()
    if not isinstance(table, pd.DataFrame) or table.empty:
        raise ValueError(f"{getter} returned no model data")
    if "Date" not in table:
        table["Date"] = pd.date_range(sim_start, periods=len(table), freq="D")
    else:
        table["Date"] = pd.to_datetime(table["Date"])
    if table.Date.min() > sim_start or table.Date.max() < required_end:
        raise ValueError(f"Model output {getter} does not cover required simulation window")
    if not table.Date.is_unique or not table.Date.is_monotonic_increasing:
        raise ValueError("AquaCrop output Date must be unique and increasing")
    return table


### 9.2 Run seven pairs of model simulations

Each pair uses common weather, soil and irrigation inputs.

In [ ]:
# 9.2 — Run seven pairs of model simulations
model_agreement_rows, normalized_comparisons, raw_observation_metrics = [], [], []
water_rows, water_daily_rows, sim_qa_rows = [], [], []
# One controlled baseline-vs-informed comparison for each growing season.
for ph in phenology_gdd.itertuples(index=False):
    season = ph.Season
    bud = pd.Timestamp(ph.Budbreak_Date)
    start_year = int(season.split("/")[0]); end_year = start_year + 1
    # Define the same simulation and evaluation window for both configurations.
    sim_start = pd.Timestamp(start_year, 4, 1)
    eval_end = pd.Timestamp(end_year, 4, 30)
    available = weather_df.loc[(weather_df.Date >= bud) & (weather_df.Date <= eval_end), "Daily_GDD"].sum()
    # This endpoint keeps each model running past the April evaluation window.
    # It is NOT a measured physiological grape-maturity date.
    technical_maturity_gdd = max(float(np.floor(available) + 5), 1946.0)
    terminal = first_date_after_target_gdd(bud, technical_maturity_gdd)
    if terminal <= eval_end:
        raise ValueError(f"{season}: technical crop maturity does not extend beyond 30 April")
    sim_end = max(eval_end, terminal)
    planting, harvest = bud.strftime("%m/%d"), terminal.strftime("%m/%d")
    # Configure identical grape physiology except the selected phenology timings.
    base_params = grape_parameters.copy()
    base_params["Maturity"] = technical_maturity_gdd
    baseline_crop = Crop("custom", planting_date=planting, harvest_date=harvest, **base_params)
    informed = copy.deepcopy(sentinel_crops[season])
    informed.Maturity = technical_maturity_gdd
    informed.planting_date, informed.harvest_date = planting, harvest
    # Compatibility convention observed in the historical notebook: custom crop
    # fields retained, while fallow crop reconstruction looks up the 'Default' type.
    # Model compatibility setting: retain the configured crop values.
    baseline_crop.Name = "Default"
    informed.Name = "Default"
    for param in ("Kcb", "CCx", "CDC", "PlantPop", "SeedSize"):
        if not np.isclose(getattr(baseline_crop, param), getattr(informed, param)):
            raise ValueError(f"{season}: unexpected baseline-vs-inform parameter mismatch for {param}")
    # Run both versions with the same weather, soil, water and irrigation settings.
    base_model = run_model_pair(sim_start, sim_end, baseline_crop)
    inf_model = run_model_pair(sim_start, sim_end, informed)
    grows = [dated_output(m,"get_crop_growth",sim_start,eval_end) for m in (base_model,inf_model)]
    fluxes = [dated_output(m,"get_water_flux",sim_start,eval_end) for m in (base_model,inf_model)]
    base_growth, inf_growth = grows
    base_flux, inf_flux = fluxes
    # Reject incomplete day-by-day output before comparing the two models.
    expected_dates = pd.date_range(bud, eval_end, freq="D")
    for t in [base_growth, inf_growth, base_flux, inf_flux]:
        slice_dates = t.loc[t.Date.between(bud,eval_end),"Date"]
        if len(slice_dates) != len(expected_dates) or not slice_dates.reset_index(drop=True).equals(pd.Series(expected_dates)):
            raise AssertionError(f"{season}: model outputs do not cover the complete budbreak–30 April evaluation window")
    # Align modelled canopy cover with the reconstructed Sentinel proxy dates.
    b = base_growth.loc[base_growth.Date.between(bud,eval_end),["Date","canopy_cover"]].rename(columns={"canopy_cover":"Baseline_CC"})
    inf = inf_growth.loc[inf_growth.Date.between(bud,eval_end),["Date","canopy_cover"]].rename(columns={"canopy_cover":"Informed_CC"})
    curve = canopy_smoothed[(canopy_smoothed.Season==season) & (canopy_smoothed.Date.between(bud,eval_end))]
    comp = curve[["Date","Sentinel_CC_Smoothed"]].merge(b,on="Date").merge(inf,on="Date")
    if len(comp) != len(curve) or len(comp) < 3:
        raise AssertionError(f"{season}: missing paired comparison dates")
    if not (comp[["Baseline_CC","Informed_CC"]].to_numpy(dtype=float) >= 0).all():
        raise ValueError(f"{season}: AquaCrop reported negative canopy")
    # Evaluate raw and normalised canopy agreement (in-sample, not field validation).
    base_raw = metrics(comp.Sentinel_CC_Smoothed,comp.Baseline_CC)
    inf_raw = metrics(comp.Sentinel_CC_Smoothed,comp.Informed_CC)
    normalized = normalized_metrics(comp.Sentinel_CC_Smoothed,comp.Baseline_CC,comp.Informed_CC)
    comp["Sentinel_Norm"],comp["Baseline_Norm"],comp["Informed_Norm"] = normalized
    base_norm = metrics(comp.Sentinel_Norm,comp.Baseline_Norm)
    inf_norm = metrics(comp.Sentinel_Norm,comp.Informed_Norm)
    comp["Season"] = season
    normalized_comparisons.append(comp)
    model_agreement_rows.append({"Season":season, "N_Smoothed":len(comp),
        **{f"Baseline_{k}":v for k,v in base_raw.items() if k!="N"},
        **{f"Informed_{k}":v for k,v in inf_raw.items() if k!="N"},
        **{f"Baseline_Norm_{k}":v for k,v in base_norm.items() if k!="N"},
        **{f"Informed_Norm_{k}":v for k,v in inf_norm.items() if k!="N"}})
    # Secondary metric at DIRECT original Sentinel acquisition dates (no interpolation).
    # A secondary check uses original acquisition dates rather than interpolated ones.
    obs = satellite_cc[(satellite_cc.Season==season)&(satellite_cc.Date.between(bud,eval_end))]
    raw_comp = obs[["Date","Sentinel_CC_Proxy"]].merge(b,on="Date").merge(inf,on="Date")
    if len(raw_comp) >= 3:
        raw_observation_metrics.append({"Season":season,"N_Raw":len(raw_comp),
            "Baseline_RMSE":metrics(raw_comp.Sentinel_CC_Proxy,raw_comp.Baseline_CC)["RMSE"],
            "Informed_RMSE":metrics(raw_comp.Sentinel_CC_Proxy,raw_comp.Informed_CC)["RMSE"]})
    # WATER: use exactly matching budbreak–April dates for BOTH configurations.
    # Compare modelled seasonal water fluxes over exactly the same date window.
    flux_season = []
    for name, f in [("Baseline",base_flux),("Informed",inf_flux)]:
        fs = f[f.Date.between(bud,eval_end)].copy()
        fs["ETa"] = fs.Tr + fs.Es
        fs["Season"], fs["Configuration"] = season, name
        if not np.isfinite(fs[["IrrDay","Tr","Es","ETa","DeepPerc","Runoff","TrPot"]].to_numpy(dtype=float)).all():
            raise ValueError(f"{season} {name}: invalid water-flux values")
        water_daily_rows.append(fs)
        flux_season.append(fs)
    row={"Season":season}
    for name, f in zip(("Baseline","Informed"),flux_season):
        for var,col in [("Irrigation","IrrDay"),("Transpiration","Tr"),("SoilEvap","Es"),("ETa","ETa"),
                        ("DeepPerc","DeepPerc"),("Runoff","Runoff")]:
            row[f"{name}_{var}_mm"] = float(f[col].sum())
        potential = float(f.TrPot.sum())
        row[f"{name}_Tr_TrPot"] = float(f.Tr.sum()/potential) if potential>0 else np.nan
    water_rows.append(row)
    # Keep a per-season record of model execution and time-window coverage.
    sim_qa_rows.append({"Season":season,"Budbreak":bud,"Evaluation_End":eval_end,"Simulation_Start":sim_start,
        "Simulation_End":sim_end,"Terminal_Date":terminal,"Technical_Maturity_GDD":technical_maturity_gdd,
        "Baseline_Days":len(base_growth),"Informed_Days":len(inf_growth),
        "Evaluation_Days":len(expected_dates),"Paired_Smoothed_Dates":len(comp)})


### 9.3 Save simulation outputs

Create the daily water, canopy comparison and runtime-quality tables.

In [ ]:
# 9.3 — Save simulation outputs
model_agreement = pd.DataFrame(model_agreement_rows)
canopy_comparisons = pd.concat(normalized_comparisons, ignore_index=True)
raw_obs_metrics = pd.DataFrame(raw_observation_metrics)
season_water = pd.DataFrame(water_rows)
water_daily = pd.concat(water_daily_rows, ignore_index=True)
simulation_qa = pd.DataFrame(sim_qa_rows)
check_seasons(simulation_qa.assign(Date=pd.Timestamp("2017-01-01")))
simulation_qa.to_csv(TABLES / "simulation_runtime_qa.csv", index=False)
water_daily.to_csv(PROCESSED / "aquacrop_daily_water_fluxes.csv", index=False)
print("All seven paired simulations completed, with full daily coverage checked.")


## 10. Canopy evaluation

Compare the simulated canopy with satellite greenness using RMSE, MAE and R².

### 10.1 Calculate canopy errors

Lower RMSE and MAE mean closer agreement; negative R² indicates weak predictive fit.

In [ ]:
# 10.1 — Calculate canopy errors
for name in ("RMSE", "MAE", "Norm_RMSE", "Norm_MAE"):
    model_agreement[f"{name}_Improvement_pct"] = [
        -safe_pct_change(new,base) for new,base in zip(model_agreement[f"Informed_{name}"],model_agreement[f"Baseline_{name}"])]
model_agreement.to_csv(TABLES / "canopy_metrics_by_season.csv", index=False)
canopy_comparisons.to_csv(TABLES / "canopy_paired_smoothed_comparisons.csv", index=False)
raw_obs_metrics.to_csv(TABLES / "canopy_raw_observation_sensitivity.csv", index=False)
summary_rows=[]
for label,suffix in [("Raw RMSE","RMSE"),("Raw MAE","MAE"),("Raw R2","R2"),
                     ("Normalized RMSE","Norm_RMSE"),("Normalized MAE","Norm_MAE"),("Normalized R2","Norm_R2")]:
    summary_rows.append({"Metric":label,
        "Literature_Mean_of_Season_Metrics":model_agreement[f"Baseline_{suffix}"].mean(),
        "Sentinel_Informed_Mean_of_Season_Metrics":model_agreement[f"Informed_{suffix}"].mean()})
# Average the seven seasonal metrics; do not pool all observation dates.
overall_summary = pd.DataFrame(summary_rows)
overall_summary.to_csv(TABLES / "canopy_metric_summary.csv", index=False)
print(overall_summary.round(3).to_string(index=False))
print("Seasons with lower informed raw RMSE:", (model_agreement.Informed_RMSE < model_agreement.Baseline_RMSE).sum(),"of 7")


### 10.2 Plot canopy comparisons

Compare the satellite proxy, baseline and Sentinel-informed trajectories.

In [ ]:
# 10.2 — Plot canopy comparisons
for season in SEASONS:
    comp = canopy_comparisons[canopy_comparisons.Season==season]
    obs = satellite_cc[satellite_cc.Season==season]
    fig,ax = plt.subplots(figsize=(11,5))
    ax.scatter(obs.Date,obs.Sentinel_CC_Proxy, s=23, alpha=.6,label="Original Sentinel observations")
    ax.plot(comp.Date,comp.Sentinel_CC_Smoothed,label="Sentinel smoothed proxy",lw=2.2)
    ax.plot(comp.Date,comp.Baseline_CC,label="Literature baseline",lw=2)
    ax.plot(comp.Date,comp.Informed_CC,label="Sentinel-informed",lw=2)
    ax.set(xlabel="Date", ylabel="Satellite proxy / AquaCrop canopy cover",title=f"Canopy comparison (distinct measurement scales): {season}",ylim=(0,1.05))
    ax.grid(alpha=.2);ax.legend(fontsize=8);fig.autofmt_xdate();fig.tight_layout()
    fig.savefig(FIGS["aquacrop"] / f"canopy_comparison_{season.replace('/','-')}.png",dpi=180)
    if season=="2023/2024":plt.show()
    else:plt.close(fig)

example=canopy_comparisons[canopy_comparisons.Season=="2023/2024"]
fig,ax=plt.subplots(figsize=(10,4.7))
for col,label in [("Sentinel_Norm","Satellite proxy"),("Baseline_Norm","Literature"),("Informed_Norm","Sentinel-informed")]:
    ax.plot(example.Date,example[col],label=label)
ax.set(xlabel="Date", ylabel="Each series / own maximum", title="2023/24 normalized canopy timing",ylim=(0,1.05))
ax.grid(alpha=.2);ax.legend();fig.autofmt_xdate();fig.tight_layout()
fig.savefig(FIGS["aquacrop"] / "normalized_canopy_2023-2024.png",dpi=180)
plt.show()


## 11. Water dynamics

Compare *simulated* irrigation, transpiration and soil evaporation.

### 11.1 Summarise water components

These values are simulated results under shared assumptions, not measured farm savings.

In [ ]:
# 11.1 — Summarise water components
for variable in ("Irrigation","Transpiration","SoilEvap","ETa"):
    season_water[variable+"_Change_pct"] = [
        safe_pct_change(new,base) for new,base in zip(
            season_water[f"Informed_{variable}_mm"],season_water[f"Baseline_{variable}_mm"])]
season_water.to_csv(TABLES / "water_dynamics_by_season.csv",index=False)
water_summary_rows=[]
for label,code in [("Irrigation (mm)","Irrigation_mm"),("Transpiration (mm)","Transpiration_mm"),
                   ("Soil evaporation (mm)","SoilEvap_mm"),("Actual ET (mm)","ETa_mm"),
                   ("Deep percolation (mm)","DeepPerc_mm"),("Runoff (mm)","Runoff_mm"),
                   ("Transpiration / Potential transpiration","Tr_TrPot")]:
    water_summary_rows.append({"Variable":label,"Literature_Mean":season_water[f"Baseline_{code}"].mean(),
                               "Informed_Mean":season_water[f"Informed_{code}"].mean()})
# These are scenario outputs, not field-measured water savings.
water_summary = pd.DataFrame(water_summary_rows)
water_summary.to_csv(TABLES / "water_dynamics_summary.csv",index=False)
print(water_summary.round(2).to_string(index=False))
print("Mean of season-by-season percentages:",
      season_water[[f"{x}_Change_pct" for x in ("Irrigation","Transpiration","SoilEvap","ETa")]].mean().round(2).to_dict())


### 11.2 Plot water-balance differences

Show the seasonal results for both model configurations.

In [ ]:
# 11.2 — Plot water-balance differences
for label,name,filename in [("Simulated transpiration","Transpiration","transpiration.png"),
                            ("Simulated soil evaporation","SoilEvap","soil_evaporation.png"),
                            ("Simulated actual ET","ETa","actual_et.png"),
                            ("Simulated irrigation","Irrigation","irrigation.png")]:
    x=np.arange(len(season_water));width=.35
    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(x-width/2,season_water[f"Baseline_{name}_mm"],width,label="Literature baseline")
    ax.bar(x+width/2,season_water[f"Informed_{name}_mm"],width,label="Sentinel-informed")
    ax.set(xticks=x,xticklabels=season_water.Season,ylabel="mm per season",title=label)
    ax.tick_params(axis="x",rotation=30);ax.grid(axis="y",alpha=.2);ax.legend();fig.tight_layout()
    fig.savefig(FIGS["water"] / filename,dpi=180)
    plt.show()


## 12. Final checks

Check the saved results and record the input files and model settings.

### 12.1 Check results and save the run manifest

Confirm seven completed pairs and record source hashes and model settings.

In [ ]:
# 12.1 — Check results and save the run manifest
assert list(model_agreement.Season) == SEASONS
assert list(season_water.Season) == SEASONS
assert len(simulation_qa)==7 and len(parameter_table)==7
for df,required in [(model_agreement,["Baseline_RMSE","Informed_RMSE","Baseline_Norm_RMSE","Informed_Norm_RMSE"]),
                    (season_water,["Baseline_Irrigation_mm","Informed_Irrigation_mm","Baseline_ETa_mm","Informed_ETa_mm"])]:
    assert np.isfinite(df[required].to_numpy(dtype=float)).all()
assert (sim_qa_rows and all(x["Paired_Smoothed_Dates"]>=3 for x in sim_qa_rows))
required_tables=["observation_gap_qa.csv","canopy_proxy_parameters.csv","sentinel_informed_crop_parameters.csv",
                 "simulation_runtime_qa.csv","canopy_metrics_by_season.csv","canopy_metric_summary.csv",
                 "water_dynamics_by_season.csv","water_dynamics_summary.csv"]
for fn in required_tables:
    if not (TABLES/fn).exists(): raise FileNotFoundError(f"Missing final table: {fn}")
manifest={
    "status":"full_aquacrop_seven_season_run_completed_and_checked",
    "satellite_source":SATELLITE_SOURCE,
    "satellite_input_file":str(selected_satellite_path.relative_to(PROJECT_ROOT)),
    "satellite_input_sha256":file_sha256(selected_satellite_path),
    "nasa_power_source_sha256":file_sha256(WEATHER_FILE),
    "n_satellite_observations":len(satellite),
    "n_growing_seasons":len(SEASONS),
    "growing_seasons":SEASONS,
    "cloud_threshold_percent":SCENE_CLOUD_THRESHOLD,
    "end_date_ee_exclusive":EE_END_EXCLUSIVE,
    "gdd_method":1,"t_base_c":T_BASE,"t_upper_c":T_UPPER,
    "canopy_proxy_settings":canopy_settings,
    "grape_kcb":grape_parameters["Kcb"],
    "software":{"python":platform.python_version(),"pandas":pd.__version__,
               "numpy":np.__version__,"aquacrop":getattr(aquacrop,"__version__","unknown"),
               "refet":getattr(refet,"__version__","unknown")},
    "interpretive_limitations":"Satellite-informed canopy agreement is in-sample; no independent vineyard field validation or actual irrigation records."
}
(PROCESSED/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
print("PASSED: seven growing seasons, paired coverage, outputs, finite metrics, manifest.")
print("Manifest:",PROCESSED/"run_manifest.json")


## 13. Review saved results for the video demo

**This is still the same notebook.** These cells read the outputs from the completed experiment, so you can show the results without rerunning all 14 model simulations. They can also be run on their own, after extracting the complete repository.

The saved results came from the earlier completed execution. The full crop model was **not rerun during this notebook cleanup**.

### 13.1 Check the completed run and growth-stage table

Show the source-data counts and the estimated seasonal development stages.

In [ ]:
# 13.1 — Open the saved run record (no AquaCrop rerun)
import json
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

# Find the project even when Jupyter starts in the notebooks folder.
_candidates = ([Path(os.environ["PIPELINE_ROOT"])] if os.environ.get("PIPELINE_ROOT") else []) + [
    Path.cwd(), Path.cwd().parent, Path("/content/sentinel2-aquacrop-hex-river-final")
]
_demo_root = next((p.resolve() for p in _candidates if (p / "data/processed/run_manifest.json").exists()), None)
if _demo_root is None:
    raise FileNotFoundError("Open the complete project folder or set PIPELINE_ROOT to its location.")

run = json.loads((_demo_root / "data/processed/run_manifest.json").read_text())
print("Completed-run status:", run["status"])
print("Growing seasons:", run["n_growing_seasons"])
print("Satellite observation dates:", run["n_satellite_observations"])

events = pd.read_csv(_demo_root / "data/processed/phenology_gdd.csv")
print("\nSeasonal growth stages (accumulated temperature):")
display(events[["Season", "Full_Budbreak_GDD", "Maximum_CC_GDD", "Senescence_Onset_GDD"]].round(1))

### 13.2 Show canopy results and the comparison graph

Explain that lower errors mean closer agreement with the satellite greenness proxy.

In [ ]:
# 13.2 — Show saved canopy results (no new simulation)
from IPython.display import Image, display

summary = pd.read_csv(_demo_root / "outputs/tables/canopy_metric_summary.csv")
print("Mean canopy metrics across seven seasons:")
display(summary.round(3))

# Show how satellite measurements were smoothed, then compare the two models.
for relative in [
    "outputs/figures/canopy/raw_vs_smoothed_2021-2022.png",
    "outputs/figures/aquacrop/canopy_comparison_2021-2022.png",
    "outputs/figures/aquacrop/normalized_canopy_2023-2024.png",
]:
    print("Figure:", Path(relative).name)
    display(Image(filename=str(_demo_root / relative), width=750))

### 13.3 Show water results and test the software

Water changes are simulated. The six software tests check the processing functions.

In [ ]:
# 13.3 — Show saved water results and run the independent software checks
import subprocess
import sys
from IPython.display import Image, display

water = pd.read_csv(_demo_root / "outputs/tables/water_dynamics_summary.csv")
print("Mean seasonal modelled water components:")
display(water.round(2))
display(Image(filename=str(_demo_root / "outputs/figures/water/soil_evaporation.png"), width=750))

# Run the real tests supplied with this repository.
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "tests/test_pipeline_utils.py"],
    cwd=_demo_root, capture_output=True, text=True
)
print("\nSoftware tests:")
print(completed.stdout.strip() or completed.stderr.strip())
if completed.returncode != 0:
    raise RuntimeError("A software test failed. Inspect the test output above.")

---
**Interpretation reminder:** Lower canopy error supports closer agreement with the satellite proxy, not independently validated grapevine growth or farm water savings. In 2017/18 a 50-day observation gap makes timing less certain.